# Time Testing
DAJones
March 2026

This notebook times the 6D algorithms I wrote for the paper, along side the algorithms that are already contained in the `clifford` package.  The inverse is not numerically verified here, we are just getting timings.  This file is kept for achival purposes.  

Some other planned tests:
- time hitzer in dimensions 1-5 alongside the 4 tested here.
- compare the acus and jones 6D algorithms with dimensional cased versions

There is also a test platform for the 4 algorithms at 6D across signature.


In [1]:
import clifford as cf

import warnings
from numba import NumbaDeprecationWarning
# Silence the noise
warnings.simplefilter('ignore', category=NumbaDeprecationWarning)

from jones_6D_inverse import jones_6D_inverse
from acus_6D_inverse import acus_6D_inverse

# 1. Create the Cl(3,3) layout
# This returns the layout object and a dictionary of the 64 basis blades
layout, blades = cf.Cl(3,3)

# 2. Update your local namespace with the blades (e1, e2, ..., e123456)
# This allows you to use 'e1' directly instead of 'blades["e1"]'
locals().update(blades)

# 3. Now you can construct your multivectors
H = 1 + 2*e14 + 3*e25 + 0.5*e123456
print(f"H is a {layout.dims}-dimensional multivector.")

H is a 6-dimensional multivector.


In [2]:
import timeit
import numpy as np

# ==================================
# 1. Setup the 6D layout
layout, blades = cf.Cl(6,0)

# ==================================
# 2. Generate a random dense multivector
# Random coefficients for all 64 elements
random_coeffs = np.random.randn(64)
A = layout.MultiVector(random_coeffs)

# ==================================
# 3. Define the benchmark functions

def run_shirokov(): # any D
    return A.shirokov_inverse()

def run_perwass(): # any D
    return A.leftLaInv()

def run_jones(): # <= 6D 
    return jones_6D_inverse( A )

def run_acus(): # <= 6D
    return acus_6D_inverse( A )

# ==================================
# 4. an initial run for the JIT compiler

run_shirokov( )
run_perwass( )
run_jones( )
run_acus( )

# 5. Timing Execution
n_it = 100 

# Pass the function name directly, NOT the result of the function call
t_shirokov = timeit.timeit(run_shirokov, number=n_it)
t_perwass  = timeit.timeit(run_perwass,  number=n_it)
t_jones    = timeit.timeit(run_jones,    number=n_it)
t_acus     = timeit.timeit(run_acus,     number=n_it)

M = 1000000
print(f"Shirokov Inverse: {M*t_shirokov/n_it:.3f} us per op")
print(f"Perwass Inverse:  {M*t_perwass/n_it:.3f} us per op")
print(f"Jones Inverse:    {M*t_jones/n_it:.3f} us per op")
print(f"Acus Inverse:     {M*t_acus/n_it:.3f} us per op")


Shirokov Inverse: 78.092 us per op
Perwass Inverse:  162.290 us per op
Jones Inverse:    100.909 us per op
Acus Inverse:     208.946 us per op


These results without jit optimization.
<pre>
Shirokov Inverse: 63.891 us per op
Perwass Inverse:  142.274 us per op
Jones Inverse:    96.275 us per op
Acus Inverse:     180.876 us per op
</pre>
